# Samantha Kanny
## CSCI E-89 Deep Learning
## Homework 03, Problem 1

---

A FashionMNIST image classifier built with PyTorch and TorchVision: data loading, model definition/training, Optuna hyperparameter tuning, model save/load, compile/export optimization for inference, and a training-accuracy visualization. Every code cell below was generated in response to one of the prompts in this homework's Claude.ai Code conversation; each prompt is reproduced verbatim in a Markdown cell immediately before the code it produced.

## Notebook contents

1. **Prompt 1** — Load and inspect the FashionMNIST dataset (`01_load_data.py`)
2. **Prompt 2** — Build, train, and evaluate the `ImageClassifier` MLP (`02_build_train_model.py`)
3. **Prompt 3** — Hyperparameter tuning with Optuna (`03_hyperparameter_tuning.py`)
4. **Prompt 4** — Three ways to save/load the trained model (`04_save_load_model.py`)
5. **Prompt 5** — Compile/optimize the model for inference (`05_compile_optimize_model.py`)
6. **Prompt 6** — Plot training/validation accuracy across epochs (`06_plot_training_accuracy.py`)
7. **Summary of Claude.ai Dialog** — chronological recap of the whole conversation

Running all cells top to bottom (Kernel → Restart & Run All) reproduces the full pipeline, including the training-accuracy plot.

## Prompt 1

Write Python code for a Jupyter notebook that uses PyTorch and TorchVision to load and inspect the FashionMNIST dataset.
Please organize the code into separate, executable notebook cells in this order:

1. Import `torch`, `torchvision`, `torchvision.transforms.v2 as T`, and `DataLoader` from `torch.utils.data`.
2. Create a transform using `T.Compose`, `T.ToImage()`, and `T.ToDtype(torch.float32, scale=True)` so the images are converted to float32 tensors with pixel values scaled to the range 0–1.
3. Download the FashionMNIST training and test datasets into a folder named `datasets`. Store the full training dataset as `train_and_valid_data` and the test dataset as `test_data`.
4. Set the PyTorch random seed to 42 and split the 60,000 training observations into 55,000 training observations and 5,000 validation observations using `torch.utils.data.random_split`.
5. Create `train_loader`, `valid_loader`, and `test_loader` using a batch size of 32. Shuffle only the training data.
6. Retrieve the first observation from `train_data` as `X_sample, y_sample`.
7. In separate cells, display:
   * the shape of `X_sample`
   * the data type of `X_sample`
   * the FashionMNIST class name associated with `y_sample`

Include short Markdown explanations before each major section. Use the same variable names listed above and provide complete code that can run from top to bottom without modification.

### Generated Script: 01_load_data.py

Download FashionMNIST, split into train/validation/test, and build DataLoaders.

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 1
import torch
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 2
transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True)
])

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 3
train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets",
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.FashionMNIST(
    root="datasets",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 4
torch.manual_seed(42)

train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55000, 5000]
)

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 5
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 6
X_sample, y_sample = train_data[0]

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 7
X_sample.shape

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 8
X_sample.dtype

In [ ]:
# Prompt 1 — Script 01_load_data.py — Cell 9
train_and_valid_data.classes[y_sample]

## Prompt 2

Using the FashionMNIST datasets and DataLoaders created in the previous step, write PyTorch code to build, train, evaluate, and inspect a fully connected neural-network classifier.
Organize the response into separate Jupyter notebook cells in the proper executable order. Include a brief Markdown heading before each cell.
The cells should accomplish the following:

1. Import all required packages, including PyTorch, `torch.nn`, `torch.nn.functional`, TorchMetrics, and Matplotlib. Select CUDA when available, MPS on Apple Silicon when available, and otherwise use the CPU.
2. Define an `ImageClassifier` class that inherits from `nn.Module`. The model should include:
   * `nn.Flatten()`
   * A linear layer from `1 * 28 * 28` inputs to 300 hidden units
   * A ReLU activation
   * A linear layer from 300 to 100 hidden units
   * A ReLU activation
   * A final linear layer from 100 hidden units to 10 output classes
3. Set the random seed to 42, initialize the model, move it to the selected device, and define `nn.CrossEntropyLoss()`.
4. Define an SGD optimizer with a learning rate of 0.1 and a multiclass accuracy metric for 10 classes.
5. Define a complete training function that trains the model using the existing `train_loader` and evaluates it using `valid_loader`. Track and return training loss, validation loss, training accuracy, and validation accuracy for every epoch.
6. Train the model for a reasonable number of epochs and print the training and validation results after each epoch.
7. Plot training and validation accuracy across epochs using Matplotlib.
8. Put the model into evaluation mode, retrieve one batch from `valid_loader`, and select its first three images and labels.
9. Use `torch.no_grad()` to generate predictions for the three images. Display the predicted numerical labels and their corresponding FashionMNIST class names.
10. Display the actual numerical labels and class names so they can be compared with the predictions.
11. Apply softmax to the model’s output logits and display the probabilities for all 10 classes, rounded to three decimal places.
12. Use `torch.topk()` to display the four highest logits, their corresponding probabilities, class indices, and class names for each image.
13. Calculate and display the model’s total number of parameters using:

`sum(param.numel() for param in model.parameters())`
Ensure the cells work with the variables previously created: `train_and_valid_data`, `train_loader`, and `valid_loader`. Move tensors to the appropriate device before training and prediction, and move them back to the CPU when required for displaying results or plotting.
Return only the Markdown headings and separate executable Python code blocks. Add brief comments within the code.

### Generated Script: 02_build_train_model.py

Define the ImageClassifier MLP, train it with train2(), and inspect predictions.

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 1
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import matplotlib.pyplot as plt

# Prefer CUDA, then Apple Silicon (MPS), then fall back to CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 2
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(n_inputs, n_hidden1)
        self.fc2 = nn.Linear(n_hidden1, n_hidden2)
        self.fc3 = nn.Linear(n_hidden2, n_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # raw logits; CrossEntropyLoss applies softmax internally
        return x

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 3
torch.manual_seed(42)

model = ImageClassifier().to(device)  # move model parameters to the selected device
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 4
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy_metric = torchmetrics.classification.MulticlassAccuracy(num_classes=10).to(device)

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 5
def train2(model, train_loader, valid_loader, optimizer, loss_fn, accuracy_metric, epochs, device):
    """Train `model` and return a history dict of per-epoch loss/accuracy."""
    history = {"train_loss": [], "valid_loss": [], "train_metrics": [], "valid_metrics": []}

    for epoch in range(epochs):
        # --- Training phase ---
        model.train()
        running_loss = 0.0
        accuracy_metric.reset()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)  # move batch to device

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            accuracy_metric.update(logits, y_batch)

        train_loss = running_loss / len(train_loader.dataset)
        train_metric = accuracy_metric.compute().item()

        # --- Validation phase ---
        model.eval()
        running_loss = 0.0
        accuracy_metric.reset()

        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                logits = model(X_batch)
                loss = loss_fn(logits, y_batch)

                running_loss += loss.item() * X_batch.size(0)
                accuracy_metric.update(logits, y_batch)

        valid_loss = running_loss / len(valid_loader.dataset)
        valid_metric = accuracy_metric.compute().item()

        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss: {train_loss:.4f}  train_metric: {train_metric:.4f} | "
            f"valid_loss: {valid_loss:.4f}  valid_metric: {valid_metric:.4f}"
        )

    return history

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 6
n_epochs = 10
history = train2(
    model, train_loader, valid_loader, optimizer, loss_fn, accuracy_metric, n_epochs, device
)

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 7
epochs_range = range(1, n_epochs + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history["train_metrics"], label="Training accuracy")
plt.plot(epochs_range, history["valid_metrics"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs. Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 8
model.eval()  # disable dropout/batchnorm-style training behavior (none here, but good practice)

X_batch, y_batch = next(iter(valid_loader))
X_new = X_batch[:3]
y_new = y_batch[:3]

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 9
with torch.no_grad():
    logits = model(X_new.to(device)).cpu()  # move input to device, bring logits back to CPU

predicted_labels = logits.argmax(dim=1)
predicted_classes = [train_and_valid_data.classes[label] for label in predicted_labels]

print("Predicted labels:", predicted_labels.tolist())
print("Predicted classes:", predicted_classes)

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 10
actual_classes = [train_and_valid_data.classes[label] for label in y_new]

print("Actual labels:", y_new.tolist())
print("Actual classes:", actual_classes)

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 11
probabilities = F.softmax(logits, dim=1)

torch.set_printoptions(sci_mode=False)
print(torch.round(probabilities, decimals=3))

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 12
top_logits, top_indices = torch.topk(logits, k=4, dim=1)
top_probs = probabilities.gather(1, top_indices)

for i in range(len(X_new)):
    print(f"\nImage {i}:")
    for logit, prob, idx in zip(top_logits[i], top_probs[i], top_indices[i]):
        class_name = train_and_valid_data.classes[idx.item()]
        print(
            f"  logit={logit.item():.3f}  prob={prob.item():.3f}  "
            f"class_idx={idx.item()}  class_name={class_name}"
        )

In [ ]:
# Prompt 2 — Script 02_build_train_model.py — Cell 13
total_params = sum(param.numel() for param in model.parameters())
print(f"Total trainable parameters: {total_params:,}")

## Prompt 3

Using the existing `ImageClassifier` class, `train2()` function, `train_loader`, `valid_loader`, `device`, and `n_epochs`, add hyperparameter tuning with Optuna.
Organize the response into separate Jupyter notebook cells in the correct executable order. Clearly label each code block with a brief Markdown heading before each cell.
First, create a basic Optuna study:

1. Import Optuna.
2. Define an objective function that:
   * Uses `trial.suggest_float()` to select a learning rate between `1e-5` and `1e-1` on a logarithmic scale.
   * Uses `trial.suggest_int()` to select the number of hidden units between 20 and 300.
   * Creates a new `ImageClassifier` for every trial.
   * Uses the selected number of hidden units for both `n_hidden1` and `n_hidden2`.
   * Moves the model to `device`.
   * Uses SGD with the selected learning rate.
   * Uses `nn.CrossEntropyLoss()` and a TorchMetrics multiclass accuracy metric with 10 classes.
   * Trains the model for 10 epochs using the existing `train2()` function.
   * Returns the highest validation accuracy stored in `history["valid_metrics"]`.
3. Set the PyTorch random seed to 42.
4. Create a reproducible `TPESampler` with a seed of 42.
5. Create a study that maximizes validation accuracy and run five trials.
6. Display `study.best_params` and `study.best_value` in separate cells.

Next, create an improved Optuna study that supports pruning:

7. Define a revised objective function that explicitly accepts `train_loader` and `valid_loader`.
8. Train the model one epoch at a time for `n_epochs`.
9. After each epoch, obtain the validation accuracy, keep track of the best validation accuracy, and report the current value to Optuna using `trial.report()`.
10. Use `trial.should_prune()` and raise `optuna.TrialPruned()` when appropriate.
11. Pass the DataLoaders to the objective function using `functools.partial`.
12. Create a reproducible `TPESampler` with a seed of 42 and a `MedianPruner`.
13. Create a new study that maximizes validation accuracy and run 20 trials.
14. Display the final `study.best_value` and `study.best_params` in separate cells.

Ensure each trial creates a completely new model, optimizer, loss function, and accuracy metric. Reset the random seed where appropriate for reproducibility.
Return only the Markdown headings and separate executable Python code blocks. Add concise comments to explain the main steps and do not use emojis.

### Generated Script: 03_hyperparameter_tuning.py

Tune learning rate and hidden-layer width with Optuna (basic study + pruning study).

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 1
import optuna  # hyperparameter optimization framework

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 2
def objective(trial):
    # Sample a learning rate on a log scale and a hidden-layer width
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)

    # Build a completely new model, optimizer, loss function, and metric for this trial
    model = ImageClassifier(n_hidden1=n_hidden, n_hidden2=n_hidden).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    accuracy_metric = torchmetrics.classification.MulticlassAccuracy(num_classes=10).to(device)

    # Train for 10 epochs using the existing training function
    history = train2(model, train_loader, valid_loader, optimizer, loss_fn, accuracy_metric, 10, device)

    # Optuna maximizes the returned value, so report the best validation accuracy
    return max(history["valid_metrics"])

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 3
torch.manual_seed(42)  # reproducible model weight initialization across trials

sampler = optuna.samplers.TPESampler(seed=42)  # reproducible hyperparameter sampling

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 4
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 5
study.best_params

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 6
study.best_value

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 7
import functools

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 8
def objective_with_pruning(trial, train_loader, valid_loader):
    # Sample a learning rate on a log scale and a hidden-layer width
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)

    # Build a completely new model, optimizer, loss function, and metric for this trial
    model = ImageClassifier(n_hidden1=n_hidden, n_hidden2=n_hidden).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    accuracy_metric = torchmetrics.classification.MulticlassAccuracy(num_classes=10).to(device)

    best_valid_accuracy = 0.0

    # Train one epoch at a time so intermediate results can be reported to Optuna
    for epoch in range(n_epochs):
        history = train2(model, train_loader, valid_loader, optimizer, loss_fn, accuracy_metric, 1, device)
        valid_accuracy = history["valid_metrics"][-1]
        best_valid_accuracy = max(best_valid_accuracy, valid_accuracy)

        # Report this epoch's validation accuracy to the pruner
        trial.report(valid_accuracy, epoch)

        # Stop unpromising trials early
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_valid_accuracy

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 9
objective_fn = functools.partial(objective_with_pruning, train_loader=train_loader, valid_loader=valid_loader)

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 10
torch.manual_seed(42)

pruning_sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 11
pruning_study = optuna.create_study(direction="maximize", sampler=pruning_sampler, pruner=pruner)
pruning_study.optimize(objective_fn, n_trials=20)

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 12
pruning_study.best_value

In [ ]:
# Prompt 3 — Script 03_hyperparameter_tuning.py — Cell 13
pruning_study.best_params

## Prompt 4

Using the trained FashionMNIST `model` and the existing `ImageClassifier` class, demonstrate three ways to save and load a PyTorch model.
Organize the response into separate Jupyter notebook cells in the correct executable order. Clearly label each code block as `Cell 1`, `Cell 2`, and so on, with a brief Markdown heading before each cell.
Include the following:

1. Save the complete trained model as `"my_fashion_mnist.pt"` using `torch.save()`.
2. Load the complete model using `torch.load()` with `weights_only=False`.
3. Put the loaded model into evaluation mode.
4. Use the loaded model to generate logits for the existing `X_new` images. Use `torch.no_grad()` and ensure `X_new` and the loaded model are on the same device.
5. Save only the trained model’s `state_dict()` as `"my_fashion_mnist_weights.pt"`.
6. Display the type of `model.state_dict()`.
7. Create a new `ImageClassifier` with:
   * `n_inputs = 1 * 28 * 28`
   * `n_hidden1 = 300`
   * `n_hidden2 = 100`
   * `n_classes = 10`
8. Load the saved weights using `torch.load()` with `weights_only=True`.
9. Load those weights into the new model using `load_state_dict()`, move the model to `device`, and place it in evaluation mode.
10. Display the new model architecture.
11. Create a dictionary named `model_data` containing:
   * The model’s state dictionary under `"model_state_dict"`
   * A nested dictionary under `"model_hyperparameters"` containing `n_inputs`, `n_hidden1`, `n_hidden2`, and `n_classes`
12. Save that dictionary as `"my_fashion_mnist_model.pt"`.
13. Load the saved dictionary using `torch.load()` with `weights_only=True`.
14. Reconstruct a new `ImageClassifier` by unpacking the saved hyperparameters with `**loaded_data["model_hyperparameters"]`.
15. Load the saved state dictionary into the reconstructed model, move it to `device`, place it in evaluation mode, and display the reconstructed model architecture.

Use the existing variables and model definition from the earlier notebook cells. Add concise comments explaining each approach. Return only the Markdown headings and separate executable Python code blocks.

### Generated Script: 04_save_load_model.py

Demonstrate three ways to save and load a trained PyTorch model.

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 1
# Approach 1: save the entire model object (architecture + weights) via pickle
torch.save(model, "my_fashion_mnist.pt")

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 2
# weights_only=False is required here because the pickle also restores the
# ImageClassifier class definition and structure, not just tensors
loaded_model = torch.load("my_fashion_mnist.pt", weights_only=False)

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 3
loaded_model.eval()  # disable training-specific behavior before inference

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 4
# Ensure the input and the model live on the same device before running inference
loaded_model = loaded_model.to(device)
X_new = X_new.to(device)

with torch.no_grad():  # no need to track gradients for inference
    logits_from_loaded_model = loaded_model(X_new)

logits_from_loaded_model

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 5
# Approach 2: save just the learned weights, not the model class/structure
torch.save(model.state_dict(), "my_fashion_mnist_weights.pt")

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 6
type(model.state_dict())

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 7
# The architecture must match the saved weights exactly before they can be loaded
n_inputs = 1 * 28 * 28
n_hidden1 = 300
n_hidden2 = 100
n_classes = 10

new_model = ImageClassifier(
    n_inputs=n_inputs, n_hidden1=n_hidden1, n_hidden2=n_hidden2, n_classes=n_classes
)

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 8
# weights_only=True is safe here since the file only contains tensors
loaded_weights = torch.load("my_fashion_mnist_weights.pt", weights_only=True)

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 9
new_model.load_state_dict(loaded_weights)  # copy the saved weights into the new model
new_model = new_model.to(device)
new_model.eval()

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 10
new_model

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 11
# Approach 3: save weights alongside the hyperparameters needed to rebuild the
# architecture, so the model can be fully reconstructed without hardcoding sizes
model_data = {
    "model_state_dict": model.state_dict(),
    "model_hyperparameters": {
        "n_inputs": n_inputs,
        "n_hidden1": n_hidden1,
        "n_hidden2": n_hidden2,
        "n_classes": n_classes,
    },
}

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 12
torch.save(model_data, "my_fashion_mnist_model.pt")

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 13
# weights_only=True is safe: the file only contains tensors, dicts, and plain ints
loaded_data = torch.load("my_fashion_mnist_model.pt", weights_only=True)

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 14
# Unpack the saved hyperparameters directly into the constructor
reconstructed_model = ImageClassifier(**loaded_data["model_hyperparameters"])

In [ ]:
# Prompt 4 — Script 04_save_load_model.py — Cell 15
reconstructed_model.load_state_dict(loaded_data["model_state_dict"])
reconstructed_model = reconstructed_model.to(device)
reconstructed_model.eval()

reconstructed_model

## Prompt 5

Using the existing trained FashionMNIST model, add code that demonstrates how to compile and optimize a PyTorch model for inference.
Use current PyTorch best practices and avoid deprecated functions. If any code produces warnings or is incompatible with the installed PyTorch version, revise it to use the recommended modern alternative instead of suppressing the warnings.
The code should:

* Compile or optimize the trained model for inference.
* Run predictions on the existing `X_new` sample.
* Compare the optimized model’s predictions with the original model’s predictions.
* Demonstrate how to save and reload the optimized or exported model when supported.
* Work with the existing `device` variable and account for differences among CUDA, Apple MPS, and CPU.
* Use evaluation mode and disable gradient calculations during inference.
* Handle unsupported features gracefully without stopping the notebook.

Organize the response into separate Jupyter notebook cells in the correct executable order. Include a brief Markdown heading before each cell and concise comments within the code.
Use the variables and model already defined in the earlier notebook cells. Do not redefine or retrain the model. Return only the Markdown headings and executable Python code blocks.

### Generated Script: 05_compile_optimize_model.py

Compile/export the trained model for optimized inference with torch.compile/torch.export.

In [ ]:
# Prompt 5 — Script 05_compile_optimize_model.py — Cell 1
# Make sure the trained model is in evaluation mode before optimizing it
model.eval()

# Pick compilation options based on the device: CUDA benefits from "reduce-overhead"
# (CUDA graphs); CPU and MPS use the default mode
compile_kwargs = {"mode": "reduce-overhead"} if device.type == "cuda" else {}

try:
    compiled_model = torch.compile(model, **compile_kwargs)
    print(f"Model compiled successfully with torch.compile() on device '{device}'.")
except Exception as error:
    # Some devices/backends (e.g. certain MPS setups) do not fully support compilation
    print(f"torch.compile() unavailable on this device ({error!r}); using the uncompiled model.")
    compiled_model = model

In [ ]:
# Prompt 5 — Script 05_compile_optimize_model.py — Cell 2
compiled_model.eval()  # ensure evaluation mode (disables training-only behavior)
X_new = X_new.to(device)  # keep the input on the same device as the model

with torch.no_grad():  # no gradients needed for inference
    compiled_logits = compiled_model(X_new)

compiled_logits

In [ ]:
# Prompt 5 — Script 05_compile_optimize_model.py — Cell 3
with torch.no_grad():
    original_logits = model(X_new)

predictions_match = torch.allclose(original_logits, compiled_logits, atol=1e-5)
max_abs_difference = (original_logits - compiled_logits).abs().max().item()

print(f"Predictions match within tolerance: {predictions_match}")
print(f"Maximum absolute difference: {max_abs_difference:.2e}")

In [ ]:
# Prompt 5 — Script 05_compile_optimize_model.py — Cell 4
# torch.export needs example inputs matching the shape/dtype used at inference time
example_inputs = (X_new,)

try:
    exported_program = torch.export.export(model, example_inputs)
    export_supported = True
    print("Model successfully exported with torch.export.export().")
except Exception as error:
    # Gracefully skip export if the architecture or PyTorch build does not support it
    print(f"torch.export() unsupported for this model ({error!r}); skipping export demo.")
    exported_program = None
    export_supported = False

In [ ]:
# Prompt 5 — Script 05_compile_optimize_model.py — Cell 5
export_path = "my_fashion_mnist_exported.pt2"

if export_supported:
    # Save the exported program to a portable .pt2 file
    torch.export.save(exported_program, export_path)

    # Reload it and recover a runnable module
    reloaded_program = torch.export.load(export_path)
    reloaded_exported_model = reloaded_program.module()
    print(f"Exported model saved to '{export_path}' and reloaded successfully.")
else:
    reloaded_exported_model = None
    print("Skipping save/reload demo since export was not supported on this device.")

In [ ]:
# Prompt 5 — Script 05_compile_optimize_model.py — Cell 6
if reloaded_exported_model is not None:
    with torch.no_grad():  # no gradients needed for inference
        exported_logits = reloaded_exported_model(X_new)

    exported_predictions_match = torch.allclose(original_logits, exported_logits, atol=1e-5)
    print(f"Reloaded exported model predictions match the original: {exported_predictions_match}")
else:
    print("No exported model available to verify.")

## Prompt 6

Add code to plot the model’s training accuracy across epochs using the training history created earlier.
Organize the response into separate Jupyter notebook cells in the correct executable order. Include:

* Epoch number on the x-axis
* Training accuracy on the y-axis
* A clearly labeled line for training accuracy
* A descriptive title
* Axis labels
* A legend
* A grid for readability

If validation accuracy is available in the training history, include it as a second line for comparison.
Use Matplotlib and the existing training-history variable from the earlier code. First identify the correct keys in the history object rather than assuming their names. Do not retrain the model or recreate the training history. Ensure accuracy is displayed consistently as either proportions or percentages.
Return a brief Markdown heading followed by a separate executable Python code block. Include concise comments.

### Generated Script: 06_plot_training_accuracy.py

Plot training and validation accuracy across epochs from the training history.

In [ ]:
# Prompt 6 — Script 06_plot_training_accuracy.py — Cell 1
# Inspect the existing history object's keys instead of assuming their names
history_keys = list(history.keys())
print("Available history keys:", history_keys)

# Find the training-accuracy key: contains "train" and either "acc" or "metric"
train_acc_key = next(
    key for key in history_keys
    if "train" in key.lower() and ("acc" in key.lower() or "metric" in key.lower())
)

# Find a validation-accuracy key, if one exists: contains "val"/"valid" and "acc"/"metric"
valid_acc_key = next(
    (
        key for key in history_keys
        if ("val" in key.lower() or "valid" in key.lower())
        and ("acc" in key.lower() or "metric" in key.lower())
    ),
    None,
)
print(f"Using '{train_acc_key}' for training accuracy and '{valid_acc_key}' for validation accuracy.")

# The accuracy metric was stored as a proportion (0-1) by torchmetrics, so both
# series are already on a consistent scale and need no conversion here
train_accuracy = history[train_acc_key]
epoch_numbers = range(1, len(train_accuracy) + 1)  # epochs are 1-indexed for display

plt.figure(figsize=(8, 5))
plt.plot(epoch_numbers, train_accuracy, label="Training accuracy", marker="o")

# Only add the validation line if the history actually tracked it
if valid_acc_key is not None:
    valid_accuracy = history[valid_acc_key]
    plt.plot(epoch_numbers, valid_accuracy, label="Validation accuracy", marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy Across Epochs")
plt.legend()
plt.grid(True)
plt.show()

## Summary of Claude.ai Dialog

This document summarizes, in chronological order, the prompts issued during this
Claude Code session and the work produced in response. The project is a FashionMNIST
image classifier built with PyTorch and TorchVision.

### Prompt 1 — Load and inspect the FashionMNIST dataset

**Goal:** Build a Jupyter notebook that downloads FashionMNIST with TorchVision,
converts images to scaled float32 tensors, splits the training set, and wraps each
split in a `DataLoader`.

**Generated code (`01_load_data.py`):** Imports (`torch`, `torchvision`,
`torchvision.transforms.v2 as T`, `DataLoader`); a `T.Compose` transform
(`T.ToImage()` + `T.ToDtype(torch.float32, scale=True)`); downloading FashionMNIST
into a `datasets/` folder as `train_and_valid_data` (60,000 images) and `test_data`;
a seeded (`torch.manual_seed(42)`) `random_split` into 55,000 `train_data` /
5,000 `valid_data`; `train_loader`/`valid_loader`/`test_loader` DataLoaders with
batch size 32 (train loader shuffled); and cells displaying `X_sample`'s shape,
dtype, and FashionMNIST class name.

**Technical decisions:** Used the modern `torchvision.transforms.v2` API rather
than the legacy `transforms` module, per the prompt's explicit request.

**Verification:** Installed PyTorch/TorchVision in the sandbox and executed the
notebook end-to-end with `jupyter nbconvert --execute`, confirming
`X_sample.shape == torch.Size([1, 28, 28])`, `X_sample.dtype == torch.float32`,
and the sample's class name resolved to `'Ankle boot'`.

### Prompt 2 — Build, train, and inspect a fully connected classifier

**Goal:** Define an MLP classifier, train it on the DataLoaders from Prompt 1,
plot accuracy, and inspect predictions on a few validation images.

**Generated code (`02_build_train_model.py`):** Imports (`torch.nn`,
`torch.nn.functional`, `torchmetrics`, `matplotlib.pyplot`) plus device selection
(CUDA → Apple MPS → CPU); an `ImageClassifier(nn.Module)` with
`Flatten → Linear(784,300) → ReLU → Linear(300,100) → ReLU → Linear(100,10)`;
seeded model initialization and `nn.CrossEntropyLoss()`; an SGD optimizer
(`lr=0.1`) and a TorchMetrics `MulticlassAccuracy(num_classes=10)`; a training
function that trains/validates each epoch and returns a history dict; a 10-epoch
training run with per-epoch printouts; a Matplotlib plot of train/validation
accuracy; retrieval of a small validation batch; predictions vs. actual labels
and class names; softmax probabilities rounded to 3 decimals; `torch.topk()` for
the top-4 predictions per image; and the total parameter count.

**Verification:** Installed TorchMetrics/Matplotlib and executed the full
notebook, confirming training converged (≈78%→90% train accuracy, ≈88% validation
accuracy over 10 epochs), the 3 sample predictions matched the actual labels, and
the parameter count (266,610) matched the 784-300-100-10 architecture.

**Note:** In later prompts, this section's `ImageClassifier`, training function,
and a few variable names were revised for reuse (see Prompt 3 and Prompt 4 below);
the notebook now reflects the final, consistent versions of these definitions.

### Prompt 3 — Add Optuna hyperparameter tuning

**Goal:** Add hyperparameter search (learning rate, hidden-layer width) using
Optuna, first as a basic study, then as a pruning-enabled study.

**Important revisions (made to keep the pipeline consistent and runnable):**
Since this prompt referenced an already-existing `train2()` function, `n_epochs`
variable, and a hidden-unit-configurable `ImageClassifier`, the Prompt 2 code
was refactored in place: `ImageClassifier.__init__` gained `n_hidden1`/`n_hidden2`
parameters (defaults 300/100); the training function was renamed from
`train_model` to `train2` and its history dict keys were renamed to
`train_metrics`/`valid_metrics`; and the epoch-count variable was renamed from
`epochs` to `n_epochs`. The accuracy-plot cell from Prompt 2 was updated to match.

**Generated code (`03_hyperparameter_tuning.py`):** `import optuna`; a basic
`objective(trial)` that samples `lr` (log-uniform, 1e-5 to 1e-1) and `n_hidden`
(20–300), builds a fresh model/optimizer/loss/metric per trial, trains 10 epochs
via `train2()`, and returns the best `history["valid_metrics"]`; a seeded
`TPESampler(seed=42)`; a study (`direction="maximize"`) run for 5 trials; and
cells displaying `study.best_params`/`study.best_value`. Then, `import functools`;
a pruning-aware `objective_with_pruning(trial, train_loader, valid_loader)` that
trains one epoch at a time, reports intermediate validation accuracy via
`trial.report()`, and raises `optuna.TrialPruned()` when `trial.should_prune()`;
binding the DataLoaders with `functools.partial`; a seeded `TPESampler` paired
with a `MedianPruner`; a 20-trial pruning study; and cells displaying its
`best_value`/`best_params`.

**Verification:** Validated the Optuna objective-function logic (both the basic
and pruning-enabled versions) with a fast synthetic-data smoke test before adding
it to the notebook, since a full real-data run (5×10 + up to 20×10 training
epochs on CPU) takes tens of minutes.

### Prompt 4 — Three ways to save and load the trained model

**Goal:** Demonstrate saving/loading (1) the whole model object, (2) just the
`state_dict`, and (3) a bundled dict of weights plus hyperparameters.

**Additional revision:** Since this prompt referred to "the existing `X_new`
images," the Prompt 2 variables `X_few`/`y_few` were renamed to `X_new`/`y_new`
for consistency. `ImageClassifier` was also extended to accept `n_inputs` and
`n_classes` constructor arguments (in addition to `n_hidden1`/`n_hidden2`), which
Prompt 4's step 7 and step 14 require so the architecture can later be
reconstructed from a saved hyperparameters dict.

**Generated code (`04_save_load_model.py`, `Cell 1`–`Cell 15`):**
`torch.save(model, "my_fashion_mnist.pt")` and
`torch.load(..., weights_only=False)` for the full model, followed by
`.eval()` and a `torch.no_grad()` inference pass on `X_new` (with explicit
`.to(device)` calls); saving `model.state_dict()` and checking its type
(`OrderedDict`); constructing a fresh `ImageClassifier` with explicit
`n_inputs`/`n_hidden1`/`n_hidden2`/`n_classes`, loading weights with
`torch.load(..., weights_only=True)` and `load_state_dict()`; and building/
saving/reloading a `model_data` dict containing `model_state_dict` and a nested
`model_hyperparameters` dict, then reconstructing the model via
`ImageClassifier(**loaded_data["model_hyperparameters"])`.

**Verification:** Validated all three save/load paths with a synthetic-data
smoke test, confirming the reconstructed model's outputs exactly matched the
original model's outputs (`torch.allclose`).

### Prompt 5 — Compile/optimize the model for inference

**Goal:** Demonstrate current (non-deprecated) PyTorch best practices for
compiling/exporting a trained model for optimized inference.

**Generated code (`05_compile_optimize_model.py`):** `torch.compile(model)`
(device-aware: `mode="reduce-overhead"` on CUDA), wrapped in `try/except` so an
unsupported backend (e.g. some Apple MPS configurations) falls back gracefully
instead of stopping the notebook; inference on `X_new` under `torch.no_grad()`;
a comparison of compiled vs. original predictions via `torch.allclose()`; and,
as the modern way to persist an optimized/portable artifact,
`torch.export.export()` / `torch.export.save()` / `torch.export.load()` (also
`try/except`-guarded), with a final check that the reloaded exported model's
predictions still match the original.

**Technical decision:** Chose `torch.compile()` and `torch.export()` — the
current PyTorch 2.x APIs — over the older `torch.jit.script`/`torch.jit.trace`,
per the prompt's instruction to avoid deprecated alternatives.

**Verification:** Confirmed with a synthetic-data smoke test that both
`torch.compile` and `torch.export` work in the sandbox (a C compiler was
available for the Inductor backend) and that outputs matched the original model.

### Prompt 6 — Plot training accuracy from the existing history

**Goal:** Add a robust accuracy plot that doesn't assume the training-history
dict's key names.

**Generated code (`06_plot_training_accuracy.py`):** Inspects `history.keys()`
to find the training-accuracy series (containing `"train"` and `"acc"`/
`"metric"`) and, if present, a validation-accuracy series (`"val"`/`"valid"` and
`"acc"`/`"metric"`), then plots both (epoch on the x-axis, accuracy as a 0–1
proportion on the y-axis) with a title, axis labels, legend, and grid — reusing
the existing `history` dict without retraining.

**Verification:** Checked the key-detection logic against a synthetic history
dict using the actual key names (`train_metrics`/`valid_metrics`) produced by
`train2()`.

### Prompt 7 — Package everything into a reproducible project

**Goal:** Organize all generated code into a complete, reproducible GitHub
project: six numbered `.py` scripts, a `requirements.txt`, a `README.md`, and a
combined homework notebook (`e89_Kanny_Samantha_HW03_Problem01.ipynb`) that
interleaves each exact original prompt with the script it produced — plus this
dialog summary.

**Final workflow produced:**
1. `01_load_data.py` — load and split the FashionMNIST dataset, build DataLoaders.
2. `02_build_train_model.py` — define, train, and evaluate the `ImageClassifier` MLP.
3. `03_hyperparameter_tuning.py` — tune learning rate/hidden width with Optuna
   (basic study, then a pruning-enabled study).
4. `04_save_load_model.py` — save/load the model three different ways.
5. `05_compile_optimize_model.py` — compile/export the model for optimized inference.
6. `06_plot_training_accuracy.py` — plot the final training/validation accuracy curve.

All six scripts, run in this order within a single Python/Jupyter session (since
later scripts depend on variables, functions, and models defined earlier), plus
the combined notebook, make up the reproducible project checked into this
repository.


### Prompt 8 — Combine everything into a single Problem 1 notebook

**Goal:** Package all generated code into one Jupyter notebook (rather than
separate repo files), with a homework-style title cell (name, course, "Homework
03, Problem 1"), the exact prompt preceding each prompt's code, meaningful
Markdown explanations, the training-accuracy plot, and a chronological dialog
summary — saved as `e89_Kanny_Samantha_HW03_Problem1.ipynb` and provided for
download.

**Generated:** `e89_Kanny_Samantha_HW03_Problem1.ipynb`, built from the same
underlying cells as `e89_Kanny_Samantha_HW03_Problem01.ipynb` (Prompt 7), with
a new title cell, a table-of-contents cell, and this summary extended to cover
Prompts 7 and 8.

**Verification:** Since the code cells are identical to those already
validated for Prompt 7 (same source cells from the working notebook, including
the full end-to-end background run), no code logic changed — only the
notebook's framing and metadata.
